# 04 - Foundry Tools without an LLM

Short examples of Speech, Translator, Language, Document Intelligence, Content Safety, and Content Understanding using dedicated SDKs.

## Before You Start

Select **Python (byom01-capability-testing)** as your Python Interpreter and use `notebooks/` as the working directory. Follow the section matching how your Azure resources were deployed.

### Deployed via Notebooks

You used this demo's deployment/configuration notebooks. If those notebooks are complete and dependency installation is verified, **no further setup is needed**. Keep the venv and generated `env/.env`. Use **Microsoft Entra ID**, leave `FOUNDRY_API_KEY` empty, and ensure your identity has the account's service data-plane permissions.

**Document Intelligence is key-only and cannot run on this key-disabled deployment.** Run the other examples independently; do not enable keys to bypass policy.

Continue at **Run the Examples**.

---

### Self deployed

Your Azure resources were deployed independently. Use the local environment from [00_setup.ipynb](./00_setup.ipynb); no Azure redeployment is needed.

If local environment is incomplete, run Setup's **Step 1**, select the demo kernel, sign in in **Step 2** for Entra or key retrieval, and install dependencies in **Step 3**. Otherwise reuse the verified venv. Populate `env/.env` as needed, using the variables below and [the environment example](../env/.env.example). Never commit credentials.

| Variable | Requirement / value |
|---|---|
| `AZURE_AI_ENDPOINT` | Required. Account endpoint, for example `https://<account>.services.ai.azure.com/`, not a project or OpenAI v1 URL. |
| `AZURE_ACCOUNT_NAME` | Required. Existing account name, also used to derive its custom Cognitive Services domain. |
| `AZURE_AI_RESOURCE_ID` | Required. Full ARM resource ID of that account; used by Translator with Entra. |
| `AZURE_LOCATION` | Required. Account region, for example `uksouth`; used by Translator. |
| `AZURE_SUBSCRIPTION_ID` | Required. Subscription containing the account; used for Document Intelligence key retrieval. |
| `AZURE_RESOURCE_GROUP` | Required. Account's resource group; used for Document Intelligence key retrieval. |
| `FOUNDRY_API_KEY` | Optional. Native key for this same account, only if local key authentication is supported and permitted. Empty selects Entra for dual-auth services. |
| `FOUNDRY_BASE_URL` | Required with `FOUNDRY_API_KEY`. OpenAI v1 base URL for the same account as `AZURE_AI_ENDPOINT`, for example `https://<account>.services.ai.azure.com/openai/v1/`. |

The SDKs use these account settings to call the service APIs. Project, gateway, and model settings are not required. Any configured key must belong to this account.

## Run the Examples

Run **Load Account Settings**, then your chosen examples. The two OCR examples also require **Load the OCR Sample**. Each service call incurs usage charges.

Calls go to the existing account's service APIs, not the Foundry project API or APIM. These examples do not call an LLM.

## Load Account Settings

Read `env/.env` and select a configured account key or DefaultAzureCredential. Each example announces the selected method without exposing credentials; failures do not retry another method. Entra requires service data-plane permissions on the account, not just project access.

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from dotenv import dotenv_values

demo_root = Path("..").resolve()
settings = dotenv_values(demo_root / "env" / ".env")
account_settings: dict[str, str] = {}
for name in (
    "AZURE_AI_ENDPOINT", "AZURE_AI_RESOURCE_ID", "AZURE_ACCOUNT_NAME",
    "AZURE_LOCATION", "AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP",
):
    value = (settings.get(name) or "").strip()
    if not value:
        raise ValueError(f"Set {name} in env/.env for the existing Foundry account.")
    account_settings[name] = value

account_endpoint = account_settings["AZURE_AI_ENDPOINT"].rstrip("/")
cognitive_endpoint = f"https://{account_settings['AZURE_ACCOUNT_NAME']}.cognitiveservices.azure.com"
account_key = (settings.get("FOUNDRY_API_KEY") or "").strip()
if account_key and urlsplit(settings.get("FOUNDRY_BASE_URL") or "").hostname != urlsplit(account_endpoint).hostname:
    raise ValueError("FOUNDRY_API_KEY must be the native key for AZURE_AI_ENDPOINT; FOUNDRY_BASE_URL must identify that same account.")
credential = AzureKeyCredential(account_key) if account_key else DefaultAzureCredential()
auth_method = "API key" if account_key else "Microsoft Entra ID (DefaultAzureCredential)"

## Speech - Synthesize a Sentence

Use the custom account domain and the SDK's TokenCredential support for Entra. Save a short WAV under `outputs/` and play it in the notebook, without requiring a microphone or default speaker.

In [ ]:
import azure.cognitiveservices.speech as speechsdk
from IPython.display import Audio, display

print(f"Using {auth_method}")
if isinstance(credential, AzureKeyCredential):
    speech_config = speechsdk.SpeechConfig(endpoint=cognitive_endpoint, key_credential=credential)
else:
    speech_config = speechsdk.SpeechConfig(endpoint=cognitive_endpoint, token_credential=credential)
    

speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"
speech_config.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm)
wav_path = demo_root / "outputs" / "foundry-tools-speech.wav"
wav_path.parent.mkdir(parents=True, exist_ok=True)
speech_synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config,
    audio_config=speechsdk.audio.AudioOutputConfig(filename=str(wav_path)),
)
speech_result = speech_synthesizer.speak_text_async("A short sentence can become speech without a language model.").get()
if speech_result.reason == speechsdk.ResultReason.Canceled:
    details = speech_result.cancellation_details
    raise RuntimeError(f"Speech synthesis canceled: {details.reason}; {details.error_details}")
if speech_result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
    raise RuntimeError(f"Speech synthesis did not complete: {speech_result.reason}")
display(Audio(filename=str(wav_path)))

## Translator - NMT Translation

Translate one sentence from English to Spanish. Explicitly select `general`, the neural machine translation (NMT) system, with fallback disabled; do not select an LLM deployment.

In [ ]:
from azure.ai.translation.text import TextTranslationClient
from azure.ai.translation.text.models import TranslateInputItem, TranslationTarget

print(f"Using {auth_method}")
with TextTranslationClient(
    endpoint="https://api.cognitive.microsofttranslator.com",
    credential=credential,
    region=account_settings["AZURE_LOCATION"],
    resource_id=account_settings["AZURE_AI_RESOURCE_ID"],
) as translator_client:
    translations = translator_client.translate([
        TranslateInputItem(
            text="The train leaves at nine in the morning.",
            language="en",
            targets=[TranslationTarget(language="es", deployment_name="general", allow_fallback=False)],
        )
    ])
print(translations[0].translations[0].text)

## Language - Detect the Language

Detect the ISO language code of one English sentence. Text Analytics can return document-level errors even when the HTTP request succeeds.

In [ ]:
from azure.ai.textanalytics import TextAnalyticsClient

print(f"Using {auth_method}")
with TextAnalyticsClient(endpoint=cognitive_endpoint, credential=credential) as language_client:
    language_result = language_client.detect_language(["The library opens early on Monday."])[0]
if language_result.is_error:
    raise RuntimeError(f"Language detection failed: {language_result.error.code}: {language_result.error.message}")
print(language_result.primary_language.iso6391_name)

## Load the OCR Sample

Fetch one public Azure sample PDF into memory for both Read examples. Do not submit customer documents.

In [ ]:
import httpx

sample_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"
sample_response = httpx.get(sample_url, timeout=60.0)
sample_response.raise_for_status()
sample_pdf = sample_response.content

## Document Intelligence - Prebuilt Read (Key Only)

Retrieve the existing account key through the management SDK using Entra, then analyze the PDF with `prebuilt-read`. Retrieval requires `Microsoft.CognitiveServices/accounts/listKeys/action`; the key is never printed or persisted.

**This multi-service route supports key-only analysis.** It requires a policy-compatible key-enabled account and is blocked on this demo's key-disabled deployment. Run the other examples independently. The notebook does not change authentication policy or retry analysis with Entra.

In [ ]:
from io import BytesIO

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

print("Using Microsoft Entra ID (DefaultAzureCredential) to retrieve the account key")
with CognitiveServicesManagementClient(
    DefaultAzureCredential(), account_settings["AZURE_SUBSCRIPTION_ID"]
) as management_client:
    document_key = management_client.accounts.list_keys(
        account_settings["AZURE_RESOURCE_GROUP"], account_settings["AZURE_ACCOUNT_NAME"]
    ).key1
if not document_key:
    raise ValueError("The existing account did not return a Document Intelligence key.")
print("Using API key for Document Intelligence analysis")
with DocumentIntelligenceClient(
    endpoint=cognitive_endpoint, credential=AzureKeyCredential(document_key)
) as document_client:
    document_result = document_client.begin_analyze_document(
        "prebuilt-read", body=BytesIO(sample_pdf), content_type="application/pdf"
    ).result()
print(document_result.content)

## Content Safety - Classify Benign Text

Display the service's category/severity results for a harmless sentence. This is direct text classification, not an agent guardrail or groundedness check.

In [ ]:
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions

print(f"Using {auth_method}")
with ContentSafetyClient(endpoint=cognitive_endpoint, credential=credential) as safety_client:
    safety_result = safety_client.analyze_text(AnalyzeTextOptions(text="I enjoyed a peaceful walk in the park."))
for category in safety_result.categories_analysis:
    print(f"{category.category}: severity {category.severity}")

## Content Understanding - Prebuilt Read

Use the stable asynchronous SDK operation and wait for completion before displaying extracted content. `prebuilt-read` requires neither a language model nor an embedding model; no default-model configuration is performed.

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient

print(f"Using {auth_method}")
with ContentUnderstandingClient(endpoint=account_endpoint, credential=credential) as understanding_client:
    understanding_result = understanding_client.begin_analyze_binary(
        analyzer_id="prebuilt-read", binary_input=sample_pdf, content_type="application/pdf"
    ).result()
print(understanding_result.contents[0].markdown)

## Finish

The PDF stays in memory. Delete only `outputs/foundry-tools-speech.wav` when you no longer need the audio. No Azure resources were created by these examples; service-side processing/retention follows each service's policy.

Continue with [05 - Direct Model API Calls](./05_direct_api.ipynb) to compare LLM request paths.